# 04 - Embeddings and Vector Storage with ChromaDB

Once documents are chunked, we need to convert them into a format that enables **semantic search**. This is where **embeddings** come in.

An embedding is a dense vector (e.g., 1536 dimensions for OpenAI's model) that captures the semantic meaning of text. Similar texts have vectors that are close together in the embedding space.

**Why not just use keyword search?**
- "What is the capital of France?" and "Paris is the main city of France" share few keywords but are semantically related.
- Embeddings capture this semantic similarity; keyword search (BM25) would miss it.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

from dotenv import load_dotenv
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

## 1. Understanding Embeddings

Let's embed a few sentences and see how similarity works.

In [ ]:
from rag_engine.embeddings.manager import get_embedding_model

embedding_model = get_embedding_model()

sentences = [
    "RAG combines retrieval with text generation.",
    "Retrieval-augmented generation enhances LLM responses.",
    "Vector databases store high-dimensional embeddings.",
    "The weather in Paris is pleasant in spring.",
    "ChromaDB is an open-source vector store.",
]

vectors = embedding_model.embed_documents(sentences)

print(f"Number of sentences: {len(vectors)}")
print(f"Embedding dimensions: {len(vectors[0])}")
print(f"First 10 values of sentence 1: {vectors[0][:10]}")

## 2. Cosine Similarity Between Embeddings

Cosine similarity measures the angle between two vectors (1.0 = identical direction, 0 = orthogonal).

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

sim_matrix = cosine_similarity(vectors)

labels = [s[:40] + "..." if len(s) > 40 else s for s in sentences]
df = pd.DataFrame(sim_matrix, index=labels, columns=labels)

print("Cosine Similarity Matrix:")
print(df.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix, cmap="YlOrRd", vmin=0.5, vmax=1.0)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels([f"S{i+1}" for i in range(len(labels))], fontsize=10)
ax.set_yticklabels([f"S{i+1}: {l[:30]}" for i, l in enumerate(labels)], fontsize=9)
plt.colorbar(im, label="Cosine Similarity")
plt.title("Embedding Similarity Heatmap")
plt.tight_layout()
plt.show()

Notice how sentences about RAG (S1, S2) are very similar to each other, and sentences about vector databases (S3, S5) cluster together. The weather sentence (S4) is distant from all others.

## 3. Storing Embeddings in ChromaDB

ChromaDB is a lightweight, local vector database. No setup, no cloud account — just `pip install chromadb`.

In [ ]:
from rag_engine.loaders import load_documents
from rag_engine.chunking.strategies import chunk_documents
from rag_engine.vectorstore.chroma_store import get_vectorstore, add_documents

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "sample")

# Load and chunk documents
md_docs = load_documents(os.path.join(DATA_DIR, "rag_survey.md"))
csv_docs = load_documents(os.path.join(DATA_DIR, "ml_glossary.csv"))

all_docs = md_docs + csv_docs
chunks = chunk_documents(all_docs, strategy="recursive", chunk_size=512, chunk_overlap=50)

print(f"Total chunks to embed: {len(chunks)}")

In [ ]:
import tempfile

# Use a temp directory for this notebook (won't pollute the project)
temp_dir = tempfile.mkdtemp()

vectorstore = add_documents(
    chunks,
    collection_name="notebook_04",
    persist_directory=temp_dir,
)

print(f"Documents stored in ChromaDB at: {temp_dir}")

## 4. Querying the Vector Store

Now we can search semantically — the query is embedded and compared to all stored chunks.

In [ ]:
query = "How does chunking affect retrieval quality?"

results = vectorstore.similarity_search_with_score(query, k=3)

for i, (doc, score) in enumerate(results, 1):
    print(f"\n--- Result {i} (distance: {score:.4f}) ---")
    print(f"Source: {doc.metadata.get('source', 'N/A')}")
    print(f"Content: {doc.page_content[:200]}...")

## 5. Metadata Filtering

ChromaDB supports filtering by metadata — useful when you want to search only within certain document types.

In [ ]:
# Only search within CSV-sourced documents
results = vectorstore.similarity_search(
    "What is an embedding?",
    k=3,
    filter={"file_type": "csv"},
)

for i, doc in enumerate(results, 1):
    print(f"Result {i}: {doc.page_content[:100]}...")
    print(f"  Source: {doc.metadata.get('source', 'N/A')}")

## 6. Visualizing the Embedding Space

Let's use t-SNE to project the high-dimensional embeddings into 2D and see how documents cluster.

In [ ]:
# Get embeddings for all chunks
chunk_texts = [c.page_content for c in chunks]
chunk_sources = [c.metadata.get("file_type", "unknown") for c in chunks]

chunk_embeddings = embedding_model.embed_documents(chunk_texts)
chunk_embeddings = np.array(chunk_embeddings)

# t-SNE projection
perplexity = min(5, len(chunk_embeddings) - 1)
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
embeddings_2d = tsne.fit_transform(chunk_embeddings)

# Plot
fig, ax = plt.subplots(figsize=(10, 7))
colors = {"markdown": "steelblue", "csv": "coral", "unknown": "gray"}

for source_type in set(chunk_sources):
    mask = [s == source_type for s in chunk_sources]
    points = embeddings_2d[mask]
    ax.scatter(points[:, 0], points[:, 1], label=source_type,
              c=colors.get(source_type, "gray"), alpha=0.7, s=60)

ax.set_title("t-SNE Visualization of Document Chunk Embeddings", fontsize=13)
ax.set_xlabel("t-SNE Dimension 1")
ax.set_ylabel("t-SNE Dimension 2")
ax.legend(title="Source Type")
plt.tight_layout()
plt.show()

## Summary

| Concept | What it does |
|---------|-------------|
| **Embeddings** | Convert text to dense vectors capturing semantic meaning |
| **Cosine similarity** | Measures how semantically similar two pieces of text are |
| **ChromaDB** | Local vector database for storing and searching embeddings |
| **Metadata filtering** | Narrow search to specific document types or sources |
| **t-SNE** | Visualization technique for high-dimensional data |

**Key insight:** The quality of your embeddings directly determines retrieval quality. Better embeddings = better RAG answers.

**Next:** [05 - Retrieval Strategies](./05_retrieval_strategies.ipynb)